In [0]:
# ============================================================
# PIPELINE CONFIGURATION
# ============================================================

dbutils.widgets.text("storage_account", "stolistdataanish")
dbutils.widgets.text("container", "olist")

storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")

BASE_PATH = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
RAW_PATH = f"{BASE_PATH}/raw"
SILVER_ORDERS_PATH = f"{BASE_PATH}/silver/orders"
GOLD_PATH = f"{BASE_PATH}/gold/daily_delivered_sales"
CHECKPOINT_PATH = f"{BASE_PATH}/control/pipeline_checkpoints"


In [0]:
spark


In [0]:
display(dbutils.fs.ls(f"{RAW_PATH}/"))


In [0]:
from pyspark.sql.types import ( StructType,StructField,StringType,IntegerType,DecimalType,TimestampType,DoubleType)

order_schema = StructType([
StructField("order_id",StringType()),
StructField("customer_id",StringType()),
StructField("order_status",StringType()),
StructField("order_purchase_timestamp",TimestampType()),
StructField("order_approved_at",TimestampType()),
StructField("order_delivered_carrier_date",TimestampType()),
StructField("order_delivered_customer_date",TimestampType()),
StructField("order_estimated_delivery_date",TimestampType())

])

customer_schema = StructType([
StructField("customer_id",StringType()),
StructField("customer_unique_id",StringType()),
StructField("customer_zip_code_prefix",StringType()),
StructField("customer_city",StringType()),
StructField("customer_state",StringType())])

order_item_schema = StructType([
StructField("order_id",StringType()),
StructField("order_item_id",IntegerType()),     
StructField("product_id",StringType()),      
StructField("seller_id",StringType()),      
StructField("shipping_limit_date",TimestampType()),  
StructField("price",DoubleType()),      
StructField("freight_value",DoubleType())

])


product_schema = StructType([
StructField( "product_id",StringType()),
StructField("product_category_name",StringType()),
StructField("product_name_lenght",IntegerType()),
StructField("product_description_lenght",IntegerType()),
StructField("product_photos_qty",IntegerType()),
StructField("product_weight_g",IntegerType()),
StructField("product_length_cm",IntegerType()),
StructField("product_height_cm",IntegerType()),
StructField("product_width_cm",IntegerType())
])



seller_schema = StructType([
StructField("seller_id",StringType()),
StructField("seller_zip_code_prefix", StringType()),
StructField("seller_city",StringType()),
StructField("seller_state",StringType())
])

payment_schema=StructType([
StructField("order_id",StringType()),
StructField("payment_sequential",IntegerType()),
StructField("payment_type",StringType()),
StructField("payment_installments", IntegerType()),
StructField("payment_value",DecimalType(18,2))
])

review_schema = StructType([
StructField("review_id",StringType()),
StructField("order_id",StringType()),
StructField("review_score",IntegerType()),
StructField("review_comment_title",StringType()),
StructField("review_comment_message",StringType()),
StructField("review_creation_date",TimestampType()),
StructField("review_answer_timestamp", TimestampType())
])

geolocation_schema = StructType([
StructField("geolocation_zip_code_prefix",StringType()),
StructField("geolocation_lat",DoubleType()),
StructField("geolocation_lng",DoubleType()),
StructField("geolocation_city",StringType()),
StructField("geolocation_state",StringType())

])

category_translation_schema = StructType([
StructField("product_category_name",StringType()),
StructField("product_category_name_english",StringType())
])


In [0]:
order_df = spark.read.format("csv").option('header',True).schema(order_schema).load(f"{RAW_PATH}/olist_orders_dataset.csv")
order_df.count()


In [0]:
order_df.printSchema()


In [0]:
order_df.show(5,truncate=False)


In [0]:
def read_csv(spark,path,schema,multiline=False):
    df = spark.read.format('csv').option('header',True).schema(schema).load(path)

    if multiline==True:
        df = spark.read.format("csv").option('header',True).option("multiline",True).schema(schema).load(path)
    return df


In [0]:
raw_path = f"{RAW_PATH}/"

olist_config = {
    "orders": {"filename": "olist_orders_dataset.csv", "schema": order_schema,"multiline":False},
    "customers": {"filename": "olist_customers_dataset.csv", "schema": customer_schema,"multiline":False},
    "geolocation": {"filename": "olist_geolocation_dataset.csv", "schema": geolocation_schema,"multiline":False},
    "order_items": {"filename": "olist_order_items_dataset.csv", "schema": order_item_schema,"multiline":False},
    "order_payment": {"filename": "olist_order_payments_dataset.csv", "schema": payment_schema,"multiline":True},
    "orders_reviews": {"filename": "olist_order_reviews_dataset.csv", "schema": review_schema,"multiline":False},
    "products": {"filename": "olist_products_dataset.csv", "schema": product_schema,"multiline":False},
    "sellers": {"filename": "olist_sellers_dataset.csv", "schema": seller_schema,"multiline":False},
    "category_name_translation": {
        "filename": "product_category_name_translation.csv",
        "schema": category_translation_schema,"multiline":False
    }
}

def ingest_olist_data(spark,raw_path):
    dataframe={}
    for name,config in olist_config.items():
            dataframe[name]=read_csv(
            spark=spark,
            path=raw_path+config["filename"],
            schema=config["schema"],
            multiline=config['multiline']
            )
    return dataframe

dataframe = ingest_olist_data(spark=spark,raw_path=raw_path)
dataframe.keys()


In [0]:
for name, df in dataframe.items():
    print(name,":", df.count())


In [0]:
dataframe['orders'].count()


In [0]:
dataframe['customers'].count()


In [0]:
dataframe['order_items'].show()


In [0]:
from pyspark.sql.functions import col,count,sum
order_items_df = dataframe['order_items'].groupBy('order_id').agg(sum(col("price")).alias("total_item_price"),sum(col("freight_value")).alias("total_freight_value"),count(col("order_item_id")).alias("item_count"))


In [0]:
order_items_df.count()


In [0]:
duplciate_order_id = order_items_df.groupBy('order_id').count().filter(col("count")>1)


In [0]:
duplciate_order_id.count()


In [0]:
dataframe['order_payment'].show(10,truncate=False)


In [0]:
payment_agg_df = dataframe['order_payment'].groupBy("order_id")\
    .agg(sum("payment_value").alias("total_payment_value"),count('payment_sequential').alias('payment_count'))


In [0]:
payment_agg_df.show()


In [0]:
duplciate_payment_id = payment_agg_df.groupBy('order_id').count().filter(col("count")>1)
duplciate_payment_id.count()


In [0]:
curated_order_df= order_df.join(order_items_df,on="order_id",how="left")


In [0]:
curated_order_df.count()


In [0]:
curated_order_df = curated_order_df.join(payment_agg_df,on="order_id",how="left")
curated_order_df.count()


In [0]:
curated_order_df = curated_order_df.join(dataframe['customers'],on='customer_id',how="left")
curated_order_df.count()


In [0]:
orphan_df = dataframe['orders'].join(dataframe['customers'],on='customer_id',how="left_anti")
orphan_df.count()


In [0]:
dataframe['orders'].printSchema()


In [0]:
invalid_approval_df = dataframe['orders'].filter(col("order_approved_at")<col("order_purchase_timestamp"))
invalid_approval_df.count()


In [0]:
invalid_delivery_df = dataframe['orders'].filter(col("order_delivered_customer_date")<col("order_delivered_carrier_date"))
invalid_delivery_df.count()


In [0]:
invalid_delivery_df.show(23)


In [0]:
validstatus = ['invoiced',"processing","shipped","unavailable","created","delivered","canceled","approved"]
invalid_order_status = dataframe['orders'].filter(~col("order_status").isin(validstatus))
invalid_order_status.count()


In [0]:
from pyspark.sql.functions import when,col,sum
null_columns=[]

for column in dataframe['orders'].columns:
    null_column = sum(when(col(column).isNull(),1).otherwise(0)).alias(f'{column}null_columns_count')
    null_columns.append(null_column)
null_dataframes = dataframe['orders'].agg(*null_columns).show()


In [0]:
dataframe['orders'].groupBy("order_status").count().show()


In [0]:
null_delivery_df = dataframe['orders'].filter(col("order_delivered_customer_date").isNull())


In [0]:
null_delivery_df.groupBy('order_status').count().show()


In [0]:
invalid_delivery_date_df = null_delivery_df.filter(col("order_status")=="delivered")


In [0]:
invalid_delivery_date_df.show()


In [0]:
invalid_delivery_date_df.count()


In [0]:
curated_order_df = curated_order_df.withColumn("faulty_data",when((col("order_status")=="delivered") & (col("order_delivered_customer_date").isNull()),True).when(col("order_delivered_customer_date")<col("order_delivered_carrier_date"),True).when(col("order_approved_at")<col("order_purchase_timestamp"),True).otherwise(False))


In [0]:
curated_order_df.filter(col("faulty_data")==True).count()


In [0]:
curated_order_df.count()


In [0]:
curated_order_df.write.format("delta").mode("overwrite").save(SILVER_ORDERS_PATH)


In [0]:
curated_orders_delta_df= spark.read.format("delta").load(SILVER_ORDERS_PATH)


In [0]:
curated_orders_delta_df.printSchema()


In [0]:
curated_orders_delta_df.count()


## Silver Delta table

Create the DeltaTable handle only after the Silver write has completed.


In [0]:
from delta.tables import DeltaTable

silver_orders_table = DeltaTable.forPath(spark, SILVER_ORDERS_PATH)
silver_order_df = silver_orders_table.toDF()


## Enable Change Data Feed

CDF is enabled on the clean Silver table so future incremental pipeline changes can be
consumed without rebuilding Gold from the full Silver dataset.

The initial Silver write is Version 0. Enabling CDF creates the next Delta commit.


In [0]:
spark.sql(f"""
ALTER TABLE delta.`{SILVER_ORDERS_PATH}`
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")


In [0]:
silver_orders_table.history() \
    .select("version", "timestamp", "operation") \
    .orderBy(col("version").desc()) \
    .show(truncate=False)


## Gold: Daily Delivered Sales

Gold includes only clean delivered orders (`faulty_data = false`).

Metrics:
- `total_delivered_orders`
- `total_delivered_value`
- `avg_delivered_order_value`


In [0]:
clean_delivered_order_df = silver_order_df.filter((col("faulty_data")==False)&(col("order_status")=="delivered"))


In [0]:
from pyspark.sql.functions import to_date
clean_delivered_order_df = clean_delivered_order_df.withColumn("order_date",to_date("order_purchase_timestamp"))
clean_delivered_order_df.select("order_purchase_timestamp",
"order_date").show()


In [0]:
daily_delivered_sales_df = clean_delivered_order_df.groupBy("order_date").agg(count("order_id").alias("total_delivered_orders"),sum("total_payment_value").alias("total_delivered_value"))
daily_delivered_sales_df = daily_delivered_sales_df.withColumn("avg_delivered_order_value",(col("total_delivered_value")/col("total_delivered_orders")))


In [0]:
daily_delivered_sales_df.agg(sum("total_delivered_orders")).show()


In [0]:
daily_delivered_sales_df.write.format("delta").mode("overwrite").save(GOLD_PATH)


In [0]:
gold_daily_delivered_sales_df_path = GOLD_PATH
gold_daily_delivered_sales_df = DeltaTable.forPath(spark,gold_daily_delivered_sales_df_path)


In [0]:
gold_daily_delivered_sales = gold_daily_delivered_sales_df.toDF()


In [0]:
print(gold_daily_delivered_sales.count())
print(gold_daily_delivered_sales.show(5))


## Final validation

The Gold order count should reconcile to the clean delivered Silver population.


In [0]:
clean_delivered_count = clean_delivered_order_df.count()
gold_delivered_count = gold_daily_delivered_sales.agg(
    sum("total_delivered_orders").alias("total")
).first()["total"]

print("Clean delivered Silver orders:", clean_delivered_count)
print("Orders represented in Gold:", gold_delivered_count)
print("Reconciled:", clean_delivered_count == gold_delivered_count)


In [0]:
latest_silver_version = (
    silver_orders_table
    .history()
    .select("version")
    .first()[0]
)

checkpoint_df = spark.createDataFrame(
    [("silver_orders_to_gold_sales", latest_silver_version)],
    ["pipeline_name", "last_processed_version"]
)


In [0]:
checkpoint_df.write.format("delta").mode("overwrite").save(CHECKPOINT_PATH)


In [0]:
silver_order_df.filter((col("faulty_data")==False) & (col("order_status")=="delivered")).show(1,truncate=False)


In [0]:
gold_daily_delivered_sales_df.toDF().filter(col("order_date")=="2017-10-02").show()


In [0]:
incoming_gold_test_df = spark.createDataFrame([("e481f51cbdc54678b7cc49136f2d6af7",50.00)],["order_id","total_payment_value"])


In [0]:
incoming_gold_test_df.show()


In [0]:
silver_orders_table.alias("target").merge(incoming_gold_test_df.alias("source"),condition="source.order_id=target.order_id").whenMatchedUpdate(set={"total_payment_value":"source.total_payment_value"}).execute()


In [0]:
checkpoint_table = DeltaTable.forPath(spark,CHECKPOINT_PATH)


In [0]:
checkpoint_read_df = checkpoint_table.toDF()

last_processed_version = (
    checkpoint_read_df
    .select("last_processed_version")
    .first()[0]
)


In [0]:
print(last_processed_version)


In [0]:
latest_silver_version = silver_orders_table.history().select("version").first()[0]


In [0]:
print(latest_silver_version)


In [0]:
if latest_silver_version > last_processed_version:

    starting_version = last_processed_version + 1

    silver_cdf_df = (
        spark.read
        .format("delta")
        .option("startingVersion", starting_version)
        .option("endingVersion", latest_silver_version)
        .option("readChangeFeed", True)
        .load(SILVER_ORDERS_PATH)
    )

    print("Processing:", starting_version, "to", latest_silver_version)

else:
    print("No new changes to process")


In [0]:
change_type_list = ["update_preimage","delete"]

gold_changes_df = silver_cdf_df.withColumn("order_count_delta",when(col("_change_type").isin(change_type_list),-1).otherwise(+1))


In [0]:
gold_changes_df = gold_changes_df.withColumn("payment_value_delta",((col("total_payment_value") )*col("order_count_delta")))


In [0]:
gold_changes_df = gold_changes_df.withColumn("order_date",to_date("order_purchase_timestamp"))


In [0]:
daily_gold_delta_df = gold_changes_df.groupBy("order_date").agg(sum(col("order_count_delta")).alias("order_count_delta"),sum(col("payment_value_delta")).alias("payment_value_delta"))


In [0]:
gold_table = DeltaTable.forPath(spark,GOLD_PATH)


In [0]:
gold_table.alias("target").merge(daily_gold_delta_df.alias("source"),condition ="target.order_date = source.order_date").whenMatchedDelete(condition="target.total_delivered_orders + source.order_count_delta = 0"
).whenMatchedUpdate(set ={
    "total_delivered_orders":"target.total_delivered_orders + source.order_count_delta",
    "total_delivered_value":"target.total_delivered_value + source.payment_value_delta",
    "avg_delivered_order_value":
        "(target.total_delivered_value + source.payment_value_delta) / (target.total_delivered_orders + source.order_count_delta)"
}).whenNotMatchedInsert(
    condition="source.order_count_delta > 0",
    values={
        "order_date": "source.order_date",
        "total_delivered_orders": "source.order_count_delta",
        "total_delivered_value": "source.payment_value_delta",
        "avg_delivered_order_value":
            "source.payment_value_delta / source.order_count_delta"
    }
).execute()


In [0]:
checkpoint_table = DeltaTable.forPath(spark, CHECKPOINT_PATH)


In [0]:
last_processed_version = checkpoint_table.toDF().select( "last_processed_version").first()[0]


In [0]:
print(last_processed_version)


In [0]:
latest_silver_version = silver_orders_table.history().select("version").first()[0]
print(latest_silver_version)


In [0]:
if latest_silver_version > last_processed_version:

    starting_version = last_processed_version + 1

    silver_cdf_df = (
        spark.read
        .format("delta")
        .option("startingVersion", starting_version)
        .option("endingVersion", latest_silver_version)
        .option("readChangeFeed", True)
        .load(SILVER_ORDERS_PATH)
    )

    print("Processing:", starting_version, "to", latest_silver_version)

else:
    print("No new changes to process")


In [0]:
silver_cdf_df.select("_change_type","_commit_version","order_id","order_status","total_payment_value","faulty_data").show(truncate=False)


In [0]:
change_type_list = ["update_preimage", "delete"]
positive_change_type_list = ["update_postimage", "insert"]

gold_changes_df = silver_cdf_df.withColumn(
    "order_count_delta",
    when(
        (col("order_status") == "delivered")
        & (col("faulty_data") == False)
        & (col("_change_type").isin(change_type_list)),
        -1
    ).when(
        (col("order_status") == "delivered")
        & (col("faulty_data") == False)
        & (col("_change_type").isin(positive_change_type_list)),
        +1
    ).otherwise(0)
)


In [0]:
gold_changes_df.select(
    "_change_type",
    "order_status",
    "faulty_data",
    "total_payment_value",
    "order_count_delta"
).show(truncate=False)


In [0]:
gold_changes_df = gold_changes_df.withColumn(
    "payment_value_delta",
    col("total_payment_value") * col("order_count_delta")
)


In [0]:
gold_changes_df = gold_changes_df.withColumn(
    "order_date",
    to_date("order_purchase_timestamp")
)


In [0]:
daily_gold_delta_df = (
    gold_changes_df
    .groupBy("order_date")
    .agg(
        sum(col("order_count_delta")).alias("order_count_delta"),
        sum(col("payment_value_delta")).alias("payment_value_delta")
    )
)


In [0]:
daily_gold_delta_df.show()


In [0]:
gold_table = DeltaTable.forPath(spark, GOLD_PATH)


In [0]:
gold_table.toDF().filter(
    col("order_date") == "2017-10-02"
).select(
    "order_date",
    "total_delivered_orders",
    "total_delivered_value",
    "avg_delivered_order_value"
).show()


In [0]:
gold_table.alias("target").merge(daily_gold_delta_df.alias("source"),condition ="target.order_date = source.order_date").whenMatchedDelete(condition="target.total_delivered_orders + source.order_count_delta = 0"
).whenMatchedUpdate(set ={
    "total_delivered_orders":"target.total_delivered_orders + source.order_count_delta",
    "total_delivered_value":"target.total_delivered_value + source.payment_value_delta",
    "avg_delivered_order_value":
        "(target.total_delivered_value + source.payment_value_delta) / (target.total_delivered_orders + source.order_count_delta)"
}).whenNotMatchedInsert(
    condition="source.order_count_delta > 0",
    values={
        "order_date": "source.order_date",
        "total_delivered_orders": "source.order_count_delta",
        "total_delivered_value": "source.payment_value_delta",
        "avg_delivered_order_value":
            "source.payment_value_delta / source.order_count_delta"
    }
).execute()


In [0]:
gold_table.toDF().filter(
    col("order_date") == "2017-10-02"
).select(
    "order_date",
    "total_delivered_orders",
    "total_delivered_value",
    "avg_delivered_order_value"
).show()


In [0]:
silver_order_df.select("order_id","order_purchase_timestamp","order_status","total_payment_value").filter(col("order_status") == "delivered").show(1,truncate=False)


In [0]:
gold_table.toDF().filter(col("order_date")== "2018-07-24").show()


In [0]:
simulate_df = spark.createDataFrame([("53cdb2fc8bc7dce0b6741e2150273451","canceled")],["order_id","order_status"])


In [0]:
silver_orders_table.alias("target").merge(simulate_df.alias("source"),condition="source.order_id=target.order_id").whenMatchedUpdate(set={
    "order_status":"source.order_status"}).execute()


In [0]:
silver_order_df.filter(col("order_id")=="53cdb2fc8bc7dce0b6741e2150273451").show()


In [0]:
silver_orders_table.history().show(2,truncate=False)


In [0]:
previous_silver_df = spark.read.format("delta").option("versionAsOf", 2).load(SILVER_ORDERS_PATH)


In [0]:
previous_silver_df.select("order_id","order_status","total_payment_value","order_purchase_timestamp").filter(col("order_id")=="53cdb2fc8bc7dce0b6741e2150273451").show(truncate=False)


In [0]:
silver_comparison_df = previous_silver_df.alias("previous_silver_table").join(silver_order_df.alias("current_silver_table"),on="order_id",how="inner")


In [0]:
silver_comparison_df.filter(col("order_id")=="53cdb2fc8bc7dce0b6741e2150273451").select("order_id",col("previous_silver_table.order_status").alias("previous_status"),col("current_silver_table.order_status").alias("current_status"),col("previous_silver_table.total_payment_value").alias("previous_payment"),col("current_silver_table.total_payment_value").alias("current_payment")).show(truncate=False)


In [0]:
removed_delivered_df = silver_comparison_df.filter((col("previous_silver_table.order_status")=="delivered") & (col("current_silver_table.order_status")!="delivered"))
removed_delivered_df.count()


In [0]:
removed_delivered_df = removed_delivered_df.withColumn("order_date",to_date(col("previous_silver_table.order_purchase_timestamp")))


In [0]:
removed_delivered_df.show()


In [0]:
from pyspark.sql.functions import lit
removed_delivered_df = removed_delivered_df.withColumn("order_count_delta",lit(-1))


In [0]:
removed_delivered_df.select("order_id","order_date","order_count_delta").show()


In [0]:
removed_delivered_df = removed_delivered_df.withColumn("payment_value_delta",(col("previous_silver_table.total_payment_value"))* col("order_count_delta")) 


In [0]:
removed_delivered_df.select(
    "order_id",
    "order_date",
    "order_count_delta",
    "payment_value_delta"
).show()


In [0]:
daily_removed_delta_df = removed_delivered_df.groupBy("order_date").agg(sum(col("order_count_delta")).alias("order_count_delta")
,sum(col("payment_value_delta")).alias("payment_value_delta"))


In [0]:
daily_removed_delta_df.show()


In [0]:
gold_table.alias("target") \
    .merge(
        daily_removed_delta_df.alias("source"),
        condition="target.order_date = source.order_date"
    ) \
    .whenMatchedDelete(
        condition="target.total_delivered_orders + source.order_count_delta = 0"
    ) \
    .whenMatchedUpdate(
        set={
            "total_delivered_orders":
                "target.total_delivered_orders + source.order_count_delta",

            "total_delivered_value":
                "target.total_delivered_value + source.payment_value_delta",

            "avg_delivered_order_value":
                "(target.total_delivered_value + source.payment_value_delta) / "
                "(target.total_delivered_orders + source.order_count_delta)"
        }
    ) \
    .execute()


In [0]:
gold_table.toDF() \
    .filter(col("order_date") == "2018-07-24") \
    .show()


In [0]:
spark.sql(f"""
DESCRIBE DETAIL delta.`{SILVER_ORDERS_PATH}`
""").select("properties").show(truncate=False)


In [0]:
cdf_df = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 3) \
    .option("endingVersion", 3) \
    .load(SILVER_ORDERS_PATH)


In [0]:
cdf_df.filter(
    col("order_id") == "53cdb2fc8bc7dce0b6741e2150273451"
).select(
    "order_id",
    "order_status",
    "total_payment_value",
    "_change_type",
    "_commit_version",
    "_commit_timestamp"
).show(truncate=False)


In [0]:
removed_delivered_cdf_df = cdf_df.filter((col("_change_type")=="update_preimage")& (col("order_status")=="delivered"))


In [0]:
removed_delivered_cdf_df.select(
    "order_id",
    "order_status",
    "total_payment_value",
    "_change_type",
    "_commit_version"
).show(truncate=False)


In [0]:
postimage_df = cdf_df.filter(col("_change_type")=="update_postimage")


In [0]:
postimage_df.show()


In [0]:
cdf_comparison_df = removed_delivered_cdf_df.alias("before").join(
    postimage_df.alias("after"),
    on=col("before.order_id")==col("after.order_id"),
    how="inner"
)


In [0]:
cdf_comparison_df.select(
    col("before.order_id").alias("order_id"),
    col("before.order_status").alias("before_status"),
    col("after.order_status").alias("after_status"),
    col("before.total_payment_value").alias("before_payment"),
    col("after.total_payment_value").alias("after_payment")
).show(truncate=False)


In [0]:
removed_delivered_transition_df = cdf_comparison_df.filter((col("before.order_status")=="delivered") & (col("after.order_status")!="delivered"))


In [0]:
removed_delivered_transition_df = removed_delivered_transition_df.withColumn(
    "order_date",to_date(col("before.order_purchase_timestamp"))
)


In [0]:
removed_delivered_transition_df = removed_delivered_transition_df.withColumn(
    "order_count_delta",lit(-1)
)


In [0]:
removed_delivered_transition_df = removed_delivered_transition_df.withColumn(
    "payment_value_delta",(col("before.total_payment_value")) * (col("order_count_delta"))
)


In [0]:
removed_delivered_transition_df.select(
    col("before.order_id").alias("order_id"),
    "order_date",
    "order_count_delta",
    "payment_value_delta",
    col("before.order_status").alias("before_status"),
    col("after.order_status").alias("after_status")
).show(truncate=False)


In [0]:
TEST_ORDER_ID = "53cdb2fc8bc7dce0b6741e2150273451"

silver_orders_table.toDF() \
    .filter(col("order_id") == TEST_ORDER_ID) \
    .select(
        "order_id",
        "order_status",
        "total_payment_value",
        "order_purchase_timestamp"
    ) \
    .show(truncate=False)


In [0]:
simulate_delivered_df = spark.createDataFrame(
    [(TEST_ORDER_ID, "delivered")],
    ["order_id", "order_status"]
)


In [0]:
silver_orders_table.alias("target") \
    .merge(
        simulate_delivered_df.alias("source"),
        condition="target.order_id = source.order_id"
    ) \
    .whenMatchedUpdate(
        set={
            "order_status": "source.order_status"
        }
    ) \
    .execute()


In [0]:
silver_orders_table.history() \
    .select("version", "timestamp", "operation") \
    .show(3, truncate=False)


In [0]:
latest_version = silver_orders_table.history(1) \
    .select("version") \
    .first()["version"]


In [0]:
cdf_delivered_df = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", latest_version) \
    .option("endingVersion", latest_version) \
    .load(SILVER_ORDERS_PATH)



In [0]:
cdf_delivered_df.filter(
    col("order_id") == TEST_ORDER_ID
).select(
    "order_id",
    "order_status",
    "total_payment_value",
    "_change_type",
    "_commit_version"
).show(truncate=False)


In [0]:
before_df = cdf_delivered_df.filter(
    col("_change_type") == "update_preimage"
)

after_df = cdf_delivered_df.filter(
    col("_change_type") == "update_postimage"
)

cdf_delivered_comparison_df = before_df.alias("before").join(
    after_df.alias("after"),
    on=col("before.order_id") == col("after.order_id"),
    how="inner"
)

added_delivered_transition_df = cdf_delivered_comparison_df.filter(
    (col("before.order_status") != "delivered") &
    (col("after.order_status") == "delivered")
)


In [0]:
added_delivered_transition_df = added_delivered_transition_df \
    .withColumn(
        "order_date",
        to_date(col("after.order_purchase_timestamp"))
    ) \
    .withColumn(
        "order_count_delta",
        lit(1)
    ) \
    .withColumn(
        "payment_value_delta",
        col("after.total_payment_value") * col("order_count_delta")
    )


In [0]:
added_delivered_transition_df.select(
    col("after.order_id").alias("order_id"),
    "order_date",
    col("before.order_status").alias("before_status"),
    col("after.order_status").alias("after_status"),
    "order_count_delta",
    "payment_value_delta"
).show(truncate=False)


In [0]:
daily_added_delta_df = added_delivered_transition_df \
    .groupBy("order_date") \
    .agg(
        sum("order_count_delta").alias("order_count_delta"),
        sum("payment_value_delta").alias("payment_value_delta")
    )


In [0]:
daily_added_delta_df.show()


In [0]:
gold_table.alias("target") \
    .merge(
        daily_added_delta_df.alias("source"),
        condition="target.order_date = source.order_date"
    ) \
    .whenMatchedUpdate(
        set={
            "total_delivered_orders":
                "target.total_delivered_orders + source.order_count_delta",

            "total_delivered_value":
                "target.total_delivered_value + source.payment_value_delta",

            "avg_delivered_order_value":
                "(target.total_delivered_value + source.payment_value_delta) / "
                "(target.total_delivered_orders + source.order_count_delta)"
        }
    ) \
    .execute()


In [0]:
gold_table.toDF() \
    .filter(col("order_date") == "2018-07-24") \
    .show()
